# 07 — Tabla Resumen de Casos y Catálogo de Elementos (Punto 1)

**Objetivo:** Generar la tabla experimental de casos de daño y el catálogo de elementos para:
1. Tabla LaTeX de resumen experimental → `manuscript_AOR/tablas/tabla_casos_experimentales.tex`
2. `cases.csv` — registro estructurado de todos los escenarios evaluados
3. `elements_catalog.csv` — catálogo de elementos con tipo y zona vertical
4. Figura de distribución de casos por tipo de daño, elemento y zona

**Datos de entrada:**
- `Resultados/abolladura_2026-02-26_05-26-02/todos_los_resultados.xlsx`
- `Resultados/corrosion_2026-02-27_06-07-17/todos_los_resultados.xlsx`
- `outputs/resultados_antiguos/.../Elemento_design_type.csv` (mapeo tipo+zona)

**Punto 1 de los 6 mínimos para AOR:**  
> *Tabla/resumen de casos de daño (cuántos escenarios, elementos, severidades). Agregar figura de colores de los elementos con sus secciones.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})
%matplotlib inline

print('✓ Librerías cargadas')

## 1. Rutas y Configuración

In [ ]:
base_path = Path.home() / 'github' / 'Proyecto-doctoral'
resultados_nuevos = base_path / 'outputs' / 'resultados_nuevos'
tablas_dir = base_path / 'manuscript_AOR' / 'tablas'
tablas_dir.mkdir(parents=True, exist_ok=True)

# Fuentes de datos
fuentes = {
    'Denting': {
        'xlsx': base_path / 'Resultados' / 'abolladura_2026-02-26_05-26-02' / 'todos_los_resultados.xlsx',
        'pct_max': 45,
        'pct_step': 5,
        'label_es': 'Abolladura',
    },
    'Corrosion': {
        'xlsx': base_path / 'Resultados' / 'corrosion_2026-02-27_06-07-17' / 'todos_los_resultados.xlsx',
        'pct_max': 90,
        'pct_step': 5,
        'label_es': 'Corrosión',
    },
}

# Catálogo de elementos (tipo + zona vertical)
elem_catalog_path = base_path / 'outputs' / 'resultados_antiguos' / 'resultados_abolladuras' / 'jupyter_notebooks' / 'Elemento_design_type.csv'

for tipo, cfg in fuentes.items():
    print(f"{'✓' if cfg['xlsx'].exists() else '✗'} {tipo}: {cfg['xlsx']}")
print(f"{'✓' if elem_catalog_path.exists() else '✗'} Catálogo: {elem_catalog_path.name}")

## 2. Cargar Catálogo de Elementos

In [ ]:
elem_catalog = pd.read_csv(elem_catalog_path)
elem_catalog.columns = ['element_id', 'member_type', 'zone']

# Orden vertical de zonas (de más profundo a más superficial)
ZONE_ORDER = ['mudline', 'sub1', 'sub2', 'sub3', 'splash']
ZONE_LABELS = {
    'mudline': 'Mudline',
    'sub1': 'Submerged (lower)',
    'sub2': 'Submerged (mid)',
    'sub3': 'Submerged (upper)',
    'splash': 'Splash Zone',
}
ZONE_COLORS = {
    'mudline': '#8B4513',
    'sub1':    '#1A6FA8',
    'sub2':    '#2196F3',
    'sub3':    '#64B5F6',
    'splash':  '#E3F2FD',
}
ELEM_COLORS = {
    'Brace':       '#E74C3C',
    'Inclined_leg': '#2980B9',
    'Beam':        '#27AE60',
}

# Normalizar valores de zona (pueden tener capitalización diferente)
elem_catalog['zone'] = elem_catalog['zone'].str.lower().str.strip()

print(f'Catálogo: {len(elem_catalog)} elementos')
print()
print('Distribución por tipo:')
print(elem_catalog['member_type'].value_counts().to_string())
print()
print('Distribución por zona:')
print(elem_catalog['zone'].value_counts().to_string())
print()
print('Distribución tipo × zona:')
print(elem_catalog.groupby(['member_type','zone']).size().unstack(fill_value=0).to_string())

## 3. Construir `cases.csv` — Registro Estructurado de Escenarios

In [ ]:
all_cases = []

for damage_type, cfg in fuentes.items():
    df = pd.read_excel(cfg['xlsx'])
    
    # Merge con catálogo para obtener member_type y zone
    df_merged = df.merge(
        elem_catalog,
        left_on='Elemento', right_on='element_id',
        how='left'
    )
    
    # Construir cases rows
    cases = pd.DataFrame({
        'case_id':           df_merged['ID'].apply(lambda x: f"{damage_type[0]}_{'%04d' % x}"),
        'damage_type':       damage_type,
        'element_id':        df_merged['Elemento'],
        'severity_pct':      df_merged['Porcentaje'],
        'member_type':       df_merged['member_type'],
        'zone':              df_merged['zone'],
        'n_elements_total':  df_merged['Elemento'].max(),  # Máximo elemento = total en este modelo
        'detection_ok':      df_merged['DeteccionOK'],
        'n_false_positives': df_merged['N_FalsosPositivos'],
    })
    all_cases.append(cases)
    
    n_elem = df_merged['Elemento'].nunique()
    n_sev = df_merged['Porcentaje'].nunique()
    print(f'{damage_type}: {n_elem} elementos × {n_sev} severidades = {len(cases):,} corridas')

cases_df = pd.concat(all_cases, ignore_index=True)

# Validaciones
assert cases_df['case_id'].nunique() == len(cases_df), 'ADVERTENCIA: case_ids no únicos'
assert cases_df['member_type'].notna().all(), f"ADVERTENCIA: {cases_df['member_type'].isna().sum()} sin tipo"

out_cases = resultados_nuevos / 'cases.csv'
cases_df.to_csv(out_cases, index=False)
print(f'\n✓ cases.csv guardado: {out_cases} ({len(cases_df):,} filas)')
print(f'\nResumen:')
print(cases_df.groupby('damage_type')['element_id'].nunique().rename('elementos'))
print(cases_df['member_type'].value_counts())

## 4. Exportar `elements_catalog.csv`

In [ ]:
# Catálogo enriquecido para figura posterior
ZONE_HEIGHT_APPROX = {   # Altura aproximada por zona (para ordenar verticalmente en figura)
    'mudline': -50,
    'sub1':    -38,
    'sub2':    -25,
    'sub3':    -12,
    'splash':   0,
}

elem_catalog_out = elem_catalog.copy()
elem_catalog_out.rename(columns={'member_type': 'Design_Type'}, inplace=True)
elem_catalog_out['zone_label'] = elem_catalog_out['zone'].map(ZONE_LABELS)
elem_catalog_out['height_m_approx'] = elem_catalog_out['zone'].map(ZONE_HEIGHT_APPROX)

out_catalog = resultados_nuevos / 'elements_catalog.csv'
elem_catalog_out.to_csv(out_catalog, index=False)
print(f'✓ elements_catalog.csv guardado: {out_catalog}')
print()
print(elem_catalog_out.to_string())

## 5. Figura — Distribución de Casos por Tipo, Elemento y Zona

Panel de 3 subplots: (a) corridas por tipo de daño, (b) distribución por tipo de elemento, (c) distribución por zona vertical.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Experimental Design: Damage Scenarios Distribution',
             fontsize=14, fontweight='bold', y=1.02)

# (a) Corridas por tipo de daño
ax = axes[0]
conteo_tipo = cases_df.groupby('damage_type').size()
bars = ax.bar(conteo_tipo.index, conteo_tipo.values,
              color=['#E74C3C', '#2980B9'], edgecolor='black', linewidth=0.8)
for bar, v in zip(bars, conteo_tipo.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{v:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_xlabel('Damage Type', fontsize=12)
ax.set_ylabel('Number of Simulation Runs', fontsize=12)
ax.set_title('(a) Runs per Damage Type', fontsize=12)
ax.set_ylim(0, max(conteo_tipo.values) * 1.15)
ax.grid(axis='y', alpha=0.3)

# (b) Elementos por tipo de miembro
ax = axes[1]
conteo_elem = elem_catalog['member_type'].value_counts()
colors_elem = [ELEM_COLORS.get(e, 'gray') for e in conteo_elem.index]
wedges, texts, autotexts = ax.pie(
    conteo_elem.values,
    labels=[f'{e}\n({v})' for e, v in zip(conteo_elem.index, conteo_elem.values)],
    colors=colors_elem,
    autopct='%1.0f%%',
    startangle=90,
    pctdistance=0.75,
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight('bold')
ax.set_title(f'(b) Elements by Member Type\n(Total: {len(elem_catalog)})', fontsize=12)

# (c) Elementos por zona vertical
ax = axes[2]
zones_present = [z for z in ZONE_ORDER if z in elem_catalog['zone'].values]
conteo_zona = elem_catalog.groupby('zone')['member_type'].value_counts().unstack(fill_value=0)
conteo_zona = conteo_zona.reindex([z for z in ZONE_ORDER if z in conteo_zona.index])

bottom = np.zeros(len(conteo_zona))
y_pos = np.arange(len(conteo_zona))
for col in conteo_zona.columns:
    ax.barh(y_pos, conteo_zona[col].values, left=bottom,
            color=ELEM_COLORS.get(col, 'gray'), label=col, edgecolor='white', linewidth=0.5)
    bottom += conteo_zona[col].values

ax.set_yticks(y_pos)
ax.set_yticklabels([ZONE_LABELS.get(z, z) for z in conteo_zona.index], fontsize=10)
ax.set_xlabel('Number of Elements', fontsize=12)
ax.set_title('(c) Elements by Structural Zone', fontsize=12)
ax.legend(fontsize=9, loc='lower right')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
out_fig = resultados_nuevos / 'experimental_design_distribution.png'
plt.savefig(out_fig, dpi=300)
plt.show()
print(f'✓ Figura guardada: {out_fig}')

## 6. Tabla Resumen Experimental — Exportar a LaTeX

Tabla de 2 partes: (A) resumen global del experimento + (B) desglose por tipo de elemento.

In [ ]:

# --- Tabla A: Resumen global ---
resumen_global = []
for damage_type, cfg in fuentes.items():
    sub = cases_df[cases_df['damage_type'] == damage_type]
    n_elem = sub['element_id'].nunique()
    n_sev = sub['severity_pct'].nunique()
    sev_min = sub['severity_pct'].min()
    sev_max = sub['severity_pct'].max()
    sev_step = sorted(sub['severity_pct'].unique())
    step = sev_step[1] - sev_step[0] if len(sev_step) > 1 else 5
    n_runs = len(sub)
    resumen_global.append({
        'Damage Type':       damage_type,
        'N Elements':        n_elem,
        'Severity Range':    f'{sev_min}--{sev_max}',
        'Severity Step':     step,
        'N Severity Levels': n_sev,
        'Total Runs':        f'{n_runs:,}',
    })

df_resumen = pd.DataFrame(resumen_global)

# LaTeX
latex_A = []
latex_A.append(r'\begin{table}[htbp]')
latex_A.append(r'  \centering')
latex_A.append(r'  \caption{Summary of the numerical experiment: damage scenarios evaluated with the ICD+GA framework.}')
latex_A.append(r'  \label{tab:casos_experimentales}')
latex_A.append(r'  \begin{tabular}{lrrrrr}')
latex_A.append(r'    \toprule')
latex_A.append(r'    Damage Type & N Elements & Severity Range [\%] & Step [\%] & Levels & Total Runs \\')
latex_A.append(r'    \midrule')
for _, row in df_resumen.iterrows():
    dam   = row['Damage Type']
    n_e   = row['N Elements']
    s_r   = row['Severity Range']
    s_stp = row['Severity Step']
    n_s   = row['N Severity Levels']
    t_r   = row['Total Runs']
    latex_A.append(f'    {dam} & {n_e} & {s_r} & {s_stp} & {n_s} & {t_r} \\\\')
latex_A.append(r'    \midrule')
total_runs = cases_df.shape[0]
latex_A.append(r'    \textbf{Total} & & & & & \textbf{' + f'{total_runs:,}' + r'} \\')
latex_A.append(r'    \bottomrule')
latex_A.append(r'  \end{tabular}')
latex_A.append(r'\end{table}')

out_tabA = tablas_dir / 'tabla_casos_experimentales.tex'
out_tabA.write_text('\n'.join(latex_A), encoding='utf-8')
print(f'✓ Tabla A guardada: {out_tabA}')
print()
print('\n'.join(latex_A))


## 7. Tabla B — Desglose por Tipo de Elemento y Zona

In [ ]:
# Tabla cruzada: tipo_daño × tipo_elemento × zona → número de corridas
cases_enriched = cases_df.copy()

desglose = cases_enriched.groupby(
    ['damage_type', 'member_type', 'zone']
).agg(
    n_elements=('element_id', 'nunique'),
    n_runs=('case_id', 'count'),
).reset_index()

# LaTeX
latex_B = []
latex_B.append(r'\begin{table}[htbp]')
latex_B.append(r'  \centering')
latex_B.append(r'  \caption{Distribution of damage scenarios by member type and structural zone.}')
latex_B.append(r'  \label{tab:desglose_casos}')
latex_B.append(r'  \begin{tabular}{llllrr}')
latex_B.append(r'    \toprule')
latex_B.append(r'    Damage Type & Member Type & Zone & & N Elements & N Runs \\')
latex_B.append(r'    \midrule')

prev_dam = None
prev_mem = None
zone_label_map = ZONE_LABELS
member_short = {'Brace': 'Brace', 'Inclined_leg': 'Inclined Leg', 'Beam': 'Beam'}

for _, row in desglose.sort_values(['damage_type','member_type','zone']).iterrows():
    dam = row['damage_type']
    mem = row['member_type']
    zon = row['zone']
    
    if prev_dam and dam != prev_dam:
        latex_B.append(r'    \midrule')
    
    dam_str = row['damage_type'] if dam != prev_dam else ''
    mem_str = member_short.get(mem, mem) if mem != prev_mem else ''
    
    latex_B.append(f"    {dam_str} & {mem_str} & {zone_label_map.get(zon, zon)} & & "
                   f"{int(row['n_elements'])} & {int(row['n_runs'])} \\\\")
    prev_dam = dam
    prev_mem = mem

latex_B.append(r'    \bottomrule')
latex_B.append(r'  \end{tabular}')
latex_B.append(r'\end{table}')

out_tabB = tablas_dir / 'tabla_desglose_casos.tex'
out_tabB.write_text('\n'.join(latex_B), encoding='utf-8')
print(f'✓ Tabla B guardada: {out_tabB}')
print()
print(desglose.to_string(index=False))

## 8. Heatmap de Casos — Visualización para el Paper

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Heatmap of Simulation Runs: Element × Severity Level',
             fontsize=13, fontweight='bold')

for ax, (damage_type, cfg) in zip(axes, fuentes.items()):
    sub = cases_df[cases_df['damage_type'] == damage_type].copy()
    # Pivot: element_id × severity_pct
    # Colorear por member_type (usando número: 1=Brace, 2=Inclined_leg, 3=Beam)
    type_map = {'Brace': 1, 'Inclined_leg': 2, 'Beam': 3}
    sub['type_num'] = sub['member_type'].map(type_map)
    
    pivot = sub.pivot_table(
        index='element_id', columns='severity_pct', values='type_num', aggfunc='first'
    )
    
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(['#E74C3C', '#2980B9', '#27AE60'])  # Brace, Leg, Beam
    
    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap, vmin=0.5, vmax=3.5,
                   interpolation='nearest')
    
    ax.set_xlabel('Damage Severity [%]', fontsize=11)
    ax.set_ylabel('Element ID', fontsize=11)
    ax.set_title(f'{damage_type} ({len(sub):,} runs)', fontsize=12, fontweight='bold')
    
    # Eje X: severidades
    sev_vals = sorted(sub['severity_pct'].unique())
    ax.set_xticks(range(len(sev_vals)))
    ax.set_xticklabels([str(s) for s in sev_vals], fontsize=8, rotation=45)
    
    # Eje Y: solo cada 10 elementos
    elem_vals = list(pivot.index)
    yticks = list(range(0, len(elem_vals), 10))
    ax.set_yticks(yticks)
    ax.set_yticklabels([str(elem_vals[i]) for i in yticks], fontsize=8)
    
    # Leyenda
    patches = [
        mpatches.Patch(color='#E74C3C', label='Brace'),
        mpatches.Patch(color='#2980B9', label='Inclined Leg'),
        mpatches.Patch(color='#27AE60', label='Beam'),
    ]
    ax.legend(handles=patches, fontsize=9, loc='upper right')

plt.tight_layout()
out_heatmap = resultados_nuevos / 'heatmap_casos_elemento_severidad.png'
plt.savefig(out_heatmap, dpi=300)
plt.show()
print(f'✓ Heatmap guardado: {out_heatmap}')

## 9. Resumen Final

In [ ]:
print('=' * 70)
print('RESUMEN DEL EXPERIMENTO NUMÉRICO')
print('=' * 70)
print()
print(f'Plataforma: Jacket de 4 piernas (Golfo de México)')
print(f'Total elementos del modelo: {len(elem_catalog)}')
print(f'  • Braces (diagonales): {(elem_catalog["member_type"]=="Brace").sum()}')
print(f'  • Inclined legs (piernas): {(elem_catalog["member_type"]=="Inclined_leg").sum()}')
print(f'  • Beams (vigas): {(elem_catalog["member_type"]=="Beam").sum()}')
print()
print(f'Algoritmo: Genetic Algorithm + Hungarian Algorithm')
print(f'Indicadores de daño fusionados: 8 (DI₁–DI₈)')
print()
for damage_type, cfg in fuentes.items():
    sub = cases_df[cases_df['damage_type'] == damage_type]
    n_sev = sub['severity_pct'].nunique()
    sev_range = f"{sub['severity_pct'].min()}–{sub['severity_pct'].max()}%"
    print(f'{damage_type}:')
    print(f'  Elementos dañados: {sub["element_id"].nunique()}')
    print(f'  Severidades evaluadas: {n_sev} ({sev_range}, paso: {cfg["pct_step"]}%)')
    print(f'  Total corridas: {len(sub):,}')
    print()
print(f'TOTAL CORRIDAS: {len(cases_df):,}')
print()
print('Archivos generados:')
print('  cases.csv                                — Registro estructurado')
print('  elements_catalog.csv                     — Catálogo de elementos')
print('  experimental_design_distribution.png     — Distribución de casos')
print('  heatmap_casos_elemento_severidad.png      — Heatmap elemento×severidad')
print('  manuscript_AOR/tablas/tabla_casos_experimentales.tex')
print('  manuscript_AOR/tablas/tabla_desglose_casos.tex')
print()
print('✅ PUNTO 1 COMPLETADO — listo para integrar en main.tex')